<a href="https://colab.research.google.com/github/DarkFromAbyss/AI_Chan/blob/main/notebooks/Data_Japanese.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Connect to drive
ROOT = '/content/gdrive'
from google.colab import drive
drive.mount(ROOT, force_remount = True)

In [ ]:
!pip install -qU jamdict deep-translator

In [ ]:
# Create Project path
import os
PROJECT_PATH = f'{ROOT}/MyDrive/AiChan'
DATA_PATH = os.path.join(PROJECT_PATH, 'japanese')
DB_PATH = f"{DATA_PATH}/collection.anki21"

os.makedirs(PROJECT_PATH, exist_ok=True)
os.makedirs(DATA_PATH, exist_ok=True)

In [ ]:
# SQL query
import sqlite3
import logging

class ConnectionError(Exception):
    pass

try:
  print(f"Connect db")
  conn = sqlite3.connect(DB_PATH)
  cursor = conn.cursor()
except ConnectionError as e :
  logging.exception("Connection error: ", e)
  raise ConnectionError(e) from e

cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
cursor.fetchall()

In [ ]:
cursor.execute("PRAGMA table_info(notes);")
cols = cursor.fetchall()
cols

In [ ]:
import pandas as pd
df = pd.read_sql_query("SELECT * FROM notes", conn)
df.head(5)

In [ ]:
df.shape

In [ ]:
df_copy = df.copy()
df_copy = df.drop(columns=['id','guid', 'mid', 'mod', 'usn', 'csum','flags', 'data'])
df_copy.head(5)

In [ ]:
df_copy.tags.value_counts()

In [ ]:
import pandas as pd
import numpy as np

df_copy['tags'] = df_copy['tags'].astype(str)
df_copy['level'] = df_copy['tags'].str.extract(r'(N[1-5](?:\+N[1-5])?)')
df_copy['Frequency_Raw'] = df_copy['tags'].str.extract(r'N[1-5].*?(中低|高|中|低)')
df_copy['Frequency_Raw'] = df_copy['Frequency_Raw'].str.strip()

freq_map = {
    '高': 'High',
    '中': 'Medium',
    '中低': 'Medium-Low',
    '低': 'Low'
}

df_copy['frequency'] = df_copy['Frequency_Raw'].map(freq_map)
df_copy['frequency'] = df_copy['frequency'].fillna('Unknow')
df_copy = df_copy.drop(columns=['Frequency_Raw'])

df_copy['is_Onomatopoeia'] = df_copy['tags'].str.contains('Onomatopoeia', na=False)
df_copy['is_Extra'] = df_copy['tags'].str.contains('外', na=False)

df_copy.head(10)

In [ ]:
df_copy.isna().sum()

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

heatmap_data = pd.crosstab(df_copy['level'], df_copy['frequency'])

try:
    freq_order = ['High', 'Medium', 'Medium-Low', 'Low', 'Unrated']
    heatmap_data = heatmap_data.reindex(columns=freq_order)
except Exception as e:
    pass


plt.figure(figsize=(10, 6))
sns.heatmap(heatmap_data, annot=True, fmt='f', cmap='Blues', linewidths=.5)

plt.title('Phân bố Tần suất Từ vựng theo Cấp độ JLPT', fontsize=14, pad=15)
plt.xlabel('Tần suất (Frequency)', fontsize=12)
plt.ylabel('Cấp độ JLPT (Level)', fontsize=12)

plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
df_copy['parsed_fields'] = df_copy['flds'].str.split('\x1f')
for i in range(3):
    print(f"\nCard number {i+1} (word: {df['sfld'].iloc[i]}):")
    fields_list = df_copy['parsed_fields'].iloc[i]
    for index, content in enumerate(fields_list):
        if content.strip():
            print(f"  - Field {index}: {content}")

In [ ]:
import pandas as pd
import re

def get_field(field_list, index):
    if isinstance(field_list, list) and len(field_list) > index:
        return field_list[index].strip()
    return ""

df_copy['typing'] = df_copy['parsed_fields'].apply(lambda x: get_field(x, 3))
df_copy['reading'] = df_copy['parsed_fields'].apply(lambda x: get_field(x, 4))
df_copy['meaning'] = df_copy['parsed_fields'].apply(lambda x: get_field(x, 7))
df_copy['example'] = df_copy['parsed_fields'].apply(lambda x: get_field(x, 11))
df_copy['relword'] = df_copy['parsed_fields'].apply(lambda x: get_field(x, 17))

df_copy.head(10)

In [ ]:
temp = df_copy.copy()
missing_meaning_mask = temp['meaning'] == ""
missing_example_mask = temp['example'] == ""
missing_reword_mask = temp['relword'] == ""

plt.bar(['Missing Meaning', 'Missing Relword', 'Missing Example'],
        [missing_meaning_mask.sum(), missing_reword_mask.sum(), missing_example_mask.sum( )])
plt.title('Missing Meaning and Example')
plt.show()

In [ ]:
import json
import glob

JSON_PATH = f'{DATA_PATH}/vocabulary/jmdict-eng-*.json'

def build_full_jmdict_dataframe():
    file_paths = glob.glob(JSON_PATH)
    if not file_paths:
        raise FileNotFoundError("Không tìm thấy file JSON.")

    file_path = file_paths[0]
    print(f"Đang xử lý toàn bộ trường dữ liệu từ: {file_path}")

    with open(file_path, 'r', encoding='utf-8') as f:
        jmdict_data = json.load(f)

    parsed_data = []

    for word in jmdict_data.get('words', []):
        entry_id = word.get('id')

        # 1. Xử lý Sense (Nghĩa, Từ loại, Ghi chú)
        all_glosses = []
        all_pos = set()
        all_misc = set()

        for sense in word.get('sense', []):
            # Lấy list các nghĩa
            gloss_texts = [g['text'] for g in sense.get('gloss', [])]
            all_glosses.append("; ".join(gloss_texts))

            # Lấy Part of Speech (Danh từ, Động từ...)
            for p in sense.get('partsOfSpeech', []):
                all_pos.add(p)

            # Lấy thông tin phụ (Ví dụ: Từ lóng, Từ cổ...)
            for m in sense.get('misc', []):
                all_misc.add(m)

        full_meaning = " | ".join(all_glosses)
        pos_str = ", ".join(list(all_pos))
        misc_str = ", ".join(list(all_misc))

        # 2. Xử lý Kanji & Common (Độ phổ biến)
        kanji_entries = word.get('kanji', [])
        if not kanji_entries:
            # Nếu không có Kanji, tạo một entry giả định dựa trên Kana
            kanji_entries = [{'text': None, 'common': False, 'tags': []}]

        kana_entries = word.get('kana', [])

        # 3. Tạo tổ hợp Mapping hoàn chỉnh
        for k_obj in kanji_entries:
            for r_obj in kana_entries:
                parsed_data.append({
                    'ID': entry_id,
                    'Kanji': k_obj['text'] if k_obj['text'] else r_obj['text'],
                    'Kana': r_obj['text'],
                    'Meaning': full_meaning,
                    'Part_of_Speech': pos_str,
                    'Misc_Info': misc_str,
                    'Is_Common': k_obj.get('common', False) or r_obj.get('common', False),
                    'Kanji_Tags': ", ".join(k_obj.get('tags', [])),
                    'Kana_Tags': ", ".join(r_obj.get('tags', []))
                })

    df = pd.DataFrame(parsed_data)
    # Loại bỏ trùng lặp nếu có
    df = df.drop_duplicates(subset=['Kanji', 'Kana', 'Meaning']).reset_index(drop=True)

    print(f"Xử lý xong! DataFrame thu được: {df.shape[0]} dòng và {df.shape[1]} cột.")
    return df

# Thực thi
df_jmdict_full = build_full_jmdict_dataframe()

# Hiển thị kết quả mẫu
display(df_jmdict_full.head(10))

In [ ]:
SAVE_PATH = f'{DATA_PATH}/vocabulary/df_copy_updated.csv'

In [ ]:
import pandas as pd
import numpy as np
from deep_translator import GoogleTranslator
from tqdm import tqdm

# Kích hoạt thanh tiến trình (progress bar) cho Pandas
tqdm.pandas(desc="Đang dịch thuật")

# 1. Khởi tạo công cụ dịch (Đang để đích là Tiếng Nhật 'ja', thay bằng 'vi' nếu bạn muốn dịch sang Tiếng Việt)
translator = GoogleTranslator(source='en', target='ja')

def translate_text(text):
    """Hàm dịch an toàn, bỏ qua nếu giá trị rỗng"""
    if pd.isna(text) or text == "":
        return text
    try:
        return translator.translate(text)
    except Exception as e:
        # Bắt lỗi ngầm để không dừng toàn bộ chương trình, chỉ in ra log
        return f"[LỖI DỊCH: {text}]"

# 2. Tìm và in ra các từ bị thiếu Meaning trong df_copy
mask_missing = df_copy['meaning'].isna() | (df_copy['meaning'] == "")
df_missing = df_copy[mask_missing].copy()

print(f"Tổng số từ bị thiếu 'meaning' trong Anki (df_copy): {len(df_missing)} từ.\n")
print("-" * 50)
print("CHI TIẾT CÁC TỪ THIẾU VÀ TRẠNG THÁI TRONG JMDICT:")
print("-" * 50)

# 3. Kiểm tra các từ thiếu này có tồn tại trong JMdict hay không
# Thực hiện Left Join tập bị thiếu với JMdict
df_check = pd.merge(
    df_missing,
    df_jmdict_full[['Kanji', 'Kana', 'Meaning']],
    left_on=['sfld', 'reading'],
    right_on=['Kanji', 'Kana'],
    how='left'
)

# In ra log đối chiếu
for index, row in df_check.iterrows():
    sfld_val = row['sfld']
    reading_val = row['reading']

    # Nếu cột 'Meaning' từ JMdict có giá trị -> Tìm thấy
    if pd.notna(row['Meaning']):
        print(f"✔️ Có trong JMdict    | Kanji: {sfld_val:<10} | Kana: {reading_val}")
    else:
        print(f"❌ KHÔNG có trong JMdict | Kanji: {sfld_val:<10} | Kana: {reading_val}")

print("-" * 50)
print("\nBẮT ĐẦU QUÁ TRÌNH BÙ KHUYẾT VÀ DỊCH THUẬT...")

# 4. Thực hiện Merge tổng thể để chuẩn bị lấp đầy dữ liệu
merged_df = pd.merge(
    df_copy,
    df_jmdict_full[['Kanji', 'Kana', 'Meaning']],
    left_on=['sfld', 'reading'],
    right_on=['Kanji', 'Kana'],
    how='left'
)

# 5. Xác định chính xác các dòng thỏa mãn: Thiếu ở df_copy VÀ Có ở JMdict
mask_to_fill = (merged_df['meaning'].isna() | (merged_df['meaning'] == "")) & merged_df['Meaning'].notna()
rows_to_process = merged_df[mask_to_fill]

print(f"Số lượng từ hợp lệ sẽ được đưa vào API dịch thuật: {len(rows_to_process)} từ.\n")

if len(rows_to_process) > 0:
    # 6. Thực hiện dịch thuật có THANH TIẾN TRÌNH bằng progress_apply thay vì apply
    translated_meanings = rows_to_process['Meaning'].progress_apply(translate_text)

    # 7. Lấp đầy dữ liệu dịch vào cột meaning gốc của df_copy
    merged_df.loc[mask_to_fill, 'meaning'] = translated_meanings

# 8. Dọn dẹp DataFrame (Xóa các cột tạm sinh ra do quá trình merge)
df_copy_updated = merged_df.drop(columns=['Kanji', 'Kana', 'Meaning'])

print("\nHoàn tất! Cột 'meaning' đã được cập nhật thành công.")
display(df_copy_updated[mask_missing].head(10)) # Hiển thị 10 từ vừa được cập nhật để kiểm tra chéo

In [ ]:
import pandas as pd
import ast
from deep_translator import GoogleTranslator
from tqdm import tqdm

# Kích hoạt thanh tiến trình
tqdm.pandas(desc="Đang dịch từ Tiếng Trung -> Tiếng Nhật")

# 1. Khởi tạo công cụ dịch
translator = GoogleTranslator(source='auto', target='ja')

def extract_and_translate_chinese_v2(row):
    """
    Xử lý cột parsed_fields từ dạng String sang List để lấy Trường 5.
    """
    raw_data = row['parsed_fields']

    # Nếu dữ liệu rỗng hoặc NaN thì bỏ qua
    if pd.isna(raw_data) or raw_data == "":
        return row['meaning']

    try:
        # Chuyển đổi chuỗi "['a', 'b', ...]" thành list ['a', 'b', ...] thực thụ
        if isinstance(raw_data, str):
            fields_list = ast.literal_eval(raw_data)
        else:
            fields_list = raw_data # Trường hợp dữ liệu đã là list sẵn

        # Kiểm tra index 5 (Trường thứ 6 trong Anki)
        if isinstance(fields_list, list) and len(fields_list) > 5:
            chinese_meaning = str(fields_list[5]).strip()
            print(f"CHINESE_MEANING: {chinese_meaning}")

            if chinese_meaning and chinese_meaning != "":
                # Gọi API dịch thuật
                return translator.translate(chinese_meaning)

    except (ValueError, SyntaxError, IndexError) as e:
        # Log lỗi nếu chuỗi không đúng định dạng list hoặc không có index 5
        return row['meaning']
    except Exception as e:
        return f"[LỖI DỊCH: {str(e)}]"

    return row['meaning']

# 2. Lọc danh sách 2030 dòng thiếu meaning
mask_still_missing = df_copy['meaning'].isna() | (df_copy['meaning'] == "")

print(f"Đang xử lý {mask_still_missing.sum()} dòng bằng cách parse String -> List...")

# 3. Áp dụng xử lý
df_copy.loc[mask_still_missing, 'meaning'] = df_copy[mask_still_missing].progress_apply(extract_and_translate_chinese_v2, axis=1)
df_copy.to_csv(SAVE_PATH, index=False)
# 4. Kiểm tra kết quả cuối cùng
final_missing = (df_copy['meaning'].isna() | (df_copy['meaning'] == "")).sum()
print(f"\n✅ Hoàn tất! Số dòng khuyết thiếu hiện tại: {final_missing}")

# Xem thử kết quả đối chiếu
display(df_copy[mask_still_missing].head(10)[['sfld', 'reading', 'meaning']])

In [ ]:
df_copy_updated.to_csv(SAVE_PATH, index=False)
df_copy_updated.head(10)

In [ ]:
df_copy = pd.read_csv(SAVE_PATH)
df_copy.head(20)

In [ ]:
df_copy.sort_values(by=['level', 'typing'], ascending=False, inplace=True)
df_copy.head(20)